# Homework 2: One Dataframe

## Question 1: [IPO] Withdrawn IPOs by Company Typ

In [1]:
import pandas as pd
import numpy as np
import io

In [3]:
url = "https://www.iposcoop.com/ipos-recently-filed/"

In [5]:
import requests

response = requests.get(url)

tables = pd.read_html(io.StringIO(response.text))

In [7]:
len(tables)

1

In [9]:
for i, table in enumerate(tables):
    print("Table", i)
    print(table.columns)
    print(table.shape)
    print()

Table 0
Index(['File Date', 'Company', 'Symbol', 'Managers', 'Shares (millions)',
       'Price Low', 'Price High', 'Est $ Vol (millions)', 'Expected To Trade',
       'SCOOP Rating'],
      dtype='object')
(500, 10)



In [11]:
df = tables[0].copy()

In [15]:
df.head()

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
0,2026-09-25,Holtec Nuclear (Withdrawn),HNUC,J.P.Morgan/Guggenheim Securities/Goldman Sachs...,50.00,$15.00,$18.00,$825.00,Withdrawn,S/O
1,2026-09-25,TCGX Acquisition Corp. II (Stock-Only),TCXB,Guggenheim Securities,10.00,$10.00,$10.00,$100.00,TBA,S/O
2,2026-09-25,TFP Group (The),TFP,Morgan Stanley/Barclays/J.P. Morgan/Deutsche B...,0.00,NaN,NaN,$100.00,TBA,S/O
3,2026-09-24,ADARx Pharmaceuticals,ADRX,J.P.Morgan/Morgan Stanley/TD Cowen/UBS Investm...,26.25,$17.00,$17.00,$446.25,Priced,S/O
4,2026-09-24,Bluerock Acquisition Corp. II,BRRKU,BTIG,15.00,$10.00,$10.00,$150.00,Priced,S/O


In [17]:
df.columns

Index(['File Date', 'Company', 'Symbol', 'Managers', 'Shares (millions)',
       'Price Low', 'Price High', 'Est $ Vol (millions)', 'Expected To Trade',
       'SCOOP Rating'],
      dtype='object')

In [19]:
withdrawn = df[df["Expected To Trade"] == "Withdrawn"].copy()

In [21]:
withdrawn.shape

(35, 10)

In [23]:
def classify_company(company):
    
    if "Technologies" in company:
        return "Technologies"
    
    elif ("Acquisition Corp" in company or
          "Acquisition Corporation" in company or
          "Corp" in company):
        return "Acquisition Corp"
    
    elif "Inc" in company or "Incorporated" in company:
        return "Inc."
    
    elif "Group" in company:
        return "Group"
    
    elif "Ltd" in company or "Limited" in company:
        return "Limited"
    
    elif "Holdings" in company or "Holding" in company:
        return "Holdings"
    
    else:
        return "Other"

In [25]:
withdrawn["Company Type"] = withdrawn["Company"].apply(classify_company)

In [27]:
withdrawn[["Company", "Company Type"]]

,Company,Company Type
0,Holtec Nuclear (Withdrawn),Other
20,Rise Smart Group (Withdrawn),Group
23,C2 Capital Group (Withdrawn),Group
35,OTSAW Ltd. (Withdrawn),Limited
41,Motive Technologies (Withdrawn),Technologies
45,Idea Tech Holding (Withdrawn),Holdings
52,Hornbeck Offshore Services (Withdrawn),Other
54,Coolbit Technologies Ltd. (Withdrawn),Technologies
61,Timber Road Acquisition Corp. (Withdrawn),Acquisition Corp
71,pan-Africa Corp. (Withdrawn),Acquisition Corp


In [29]:
def extract_price(value):
    
    if pd.isna(value) or value == "-":
        return np.nan
    
    value = str(value)
    value = value.replace("$", "")
    value = value.replace(",", "")
    value = value.strip()
    
    return float(value)

In [33]:
withdrawn["Price Low Numeric"] = withdrawn["Price Low"].apply(extract_price)

In [37]:
withdrawn[["Price Low", "Price Low Numeric"]].head()

,Price Low,Price Low Numeric
0,$15.00,15.0
20,$4.00,4.0
23,$4.00,4.0
35,$4.50,4.5
41,NaN,NaN


In [39]:
withdrawn["Price High Numeric"] = withdrawn["Price High"].apply(extract_price)

In [41]:
withdrawn[[
    "Price Low",
    "Price Low Numeric",
    "Price High",
    "Price High Numeric"
]].head()

,Price Low,Price Low Numeric,Price High,Price High Numeric
0,$15.00,15.0,$18.00,18.0
20,$4.00,4.0,$4.00,4.0
23,$4.00,4.0,$5.00,5.0
35,$4.50,4.5,$5.50,5.5
41,NaN,NaN,NaN,NaN


In [43]:
withdrawn["Avg_price"] = (
    withdrawn["Price Low Numeric"] +
    withdrawn["Price High Numeric"]
) / 2

In [47]:
withdrawn[[
    "Price Low Numeric",
    "Price High Numeric",
    "Avg_price"
]].head()

,Price Low Numeric,Price High Numeric,Avg_price
0,15.0,18.0,16.5
20,4.0,4.0,4.0
23,4.0,5.0,4.5
35,4.5,5.5,5.0
41,NaN,NaN,NaN


In [49]:
def clean_numeric(value):
    
    if pd.isna(value) or value == "-":
        return np.nan
    
    value = str(value)
    value = value.replace("$", "")
    value = value.replace(",", "")
    value = value.strip()
    
    return float(value)

In [51]:
withdrawn["Shares (millions)"] = (
    withdrawn["Shares (millions)"].apply(clean_numeric)
)

In [56]:
withdrawn["Est $ Vol (millions)"].head()

0     $825.00
20      $7.20
23     $16.88
35     $22.00
41    $100.00
Name: Est $ Vol (millions), dtype: object

In [60]:
withdrawn["Calculated Value"] = (
    withdrawn["Shares (millions)"] *
    withdrawn["Avg_price"]
)

In [62]:
withdrawn[[
    "Shares (millions)",
    "Avg_price",
    "Calculated Value"
]].head()

,Shares (millions),Avg_price,Calculated Value
0,50.00,16.5,825.000
20,1.80,4.0,7.200
23,3.75,4.5,16.875
35,4.40,5.0,22.000
41,0.00,NaN,NaN


In [64]:
withdrawn["Shares_offered_value"] = (
    withdrawn["Calculated Value"]
    .fillna(withdrawn["Est $ Vol (millions)"])
)

In [66]:
withdrawn[[
    "Company",
    "Shares (millions)",
    "Avg_price",
    "Calculated Value",
    "Est $ Vol (millions)",
    "Shares_offered_value"
]].head(10)

,Company,Shares (millions),Avg_price,Calculated Value,Est $ Vol (millions),Shares_offered_value
0,Holtec Nuclear (Withdrawn),50.00,16.5,825.000,$825.00,825.0
20,Rise Smart Group (Withdrawn),1.80,4.0,7.200,$7.20,7.2
23,C2 Capital Group (Withdrawn),3.75,4.5,16.875,$16.88,16.875
35,OTSAW Ltd. (Withdrawn),4.40,5.0,22.000,$22.00,22.0
41,Motive Technologies (Withdrawn),0.00,NaN,NaN,$100.00,$100.00
45,Idea Tech Holding (Withdrawn),2.00,4.5,9.000,$9.00,9.0
52,Hornbeck Offshore Services (Withdrawn),0.00,NaN,NaN,$100.00,$100.00
54,Coolbit Technologies Ltd. (Withdrawn),5.00,4.5,22.500,$22.50,22.5
61,Timber Road Acquisition Corp. (Withdrawn),20.00,10.0,200.000,$200.00,200.0
71,pan-Africa Corp. (Withdrawn),12.50,10.0,125.000,$125.00,125.0


In [71]:
summary = (
    withdrawn
    .groupby("Company Type")["Shares_offered_value"]
    .sum()
)

TypeError: unsupported operand type(s) for +: 'float' and 'str'

In [ ]:
summary

In [73]:
withdrawn["Shares_offered_value"].dtype

dtype('O')

In [75]:
withdrawn["Shares_offered_value"].apply(type).value_counts()

Shares_offered_value
<class 'float'>    29
<class 'str'>       6
Name: count, dtype: int64

In [77]:
withdrawn[
    withdrawn["Shares_offered_value"].apply(lambda x: isinstance(x, str))
][["Company", "Shares_offered_value"]]

,Company,Shares_offered_value
41,Motive Technologies (Withdrawn),$100.00
52,Hornbeck Offshore Services (Withdrawn),$100.00
187,Kilcoy Global Foods Ltd. (Withdrawn),$100.00
276,Xinxu Copper Industry Technology Ltd.,$15.00
336,Going International Holding Company Limited,$20.00
374,Invea Therapeutics (Withdrawn),$75.00


In [79]:
withdrawn["Shares_offered_value"] = pd.to_numeric(
    withdrawn["Shares_offered_value"],
    errors="coerce"
)

In [81]:
withdrawn["Shares_offered_value"].dtype

dtype('float64')

In [83]:
withdrawn["Shares_offered_value"].isna().sum()

6

In [85]:
summary = (
    withdrawn
    .groupby("Company Type")["Shares_offered_value"]
    .sum()
    .sort_values(ascending=False)
)

summary

Company Type
Other               940.4450
Acquisition Corp    499.9850
Inc.                351.0000
Holdings            311.6575
Technologies         84.9000
Limited              84.2500
Group                56.5750
Name: Shares_offered_value, dtype: float64

In [87]:
withdrawn[
    withdrawn["Shares_offered_value"].isna()
][[
    "Company",
    "Company Type",
    "Price Low",
    "Price High",
    "Avg_price",
    "Shares (millions)",
    "Est $ Vol (millions)",
    "Shares_offered_value"
]]

,Company,Company Type,Price Low,Price High,Avg_price,Shares (millions),Est $ Vol (millions),Shares_offered_value
41,Motive Technologies (Withdrawn),Technologies,NaN,NaN,NaN,0.0,$100.00,NaN
52,Hornbeck Offshore Services (Withdrawn),Other,NaN,NaN,NaN,0.0,$100.00,NaN
187,Kilcoy Global Foods Ltd. (Withdrawn),Limited,NaN,NaN,NaN,0.0,$100.00,NaN
276,Xinxu Copper Industry Technology Ltd.,Limited,NaN,NaN,NaN,0.0,$15.00,NaN
336,Going International Holding Company Limited,Limited,NaN,NaN,NaN,0.0,$20.00,NaN
374,Invea Therapeutics (Withdrawn),Other,NaN,NaN,NaN,0.0,$75.00,NaN


In [89]:
print("Number of withdrawn IPOs:", len(withdrawn))

Number of withdrawn IPOs: 35


In [91]:
print("Missing Shares_offered_value:", withdrawn["Shares_offered_value"].isna().sum())

Missing Shares_offered_value: 6


In [93]:
summary = (
    withdrawn
    .groupby("Company Type")["Shares_offered_value"]
    .sum()
    .sort_values(ascending=False)
)

summary

Company Type
Other               940.4450
Acquisition Corp    499.9850
Inc.                351.0000
Holdings            311.6575
Technologies         84.9000
Limited              84.2500
Group                56.5750
Name: Shares_offered_value, dtype: float64

In [95]:
print("Company Type:", summary.idxmax())
print("Total Withdrawn IPO Value:", summary.max(), "million")

Company Type: Other
Total Withdrawn IPO Value: 940.445 million


In [97]:
print("Company Type:", summary.idxmax())
print("Total Withdrawn IPO Value:", summary.max(), "million")

Company Type: Other
Total Withdrawn IPO Value: 940.445 million


In [101]:
summary = summary.sort_values(ascending=False)
summary

Company Type
Other               940.4450
Acquisition Corp    499.9850
Inc.                351.0000
Holdings            311.6575
Technologies         84.9000
Limited              84.2500
Group                56.5750
Name: Shares_offered_value, dtype: float64

In [103]:
highest_company_type = summary.idxmax()

print(highest_company_type)

Other


In [105]:
highest_value = summary.max()

print(highest_value)

940.445


In [107]:
print("Company Type:", summary.idxmax())
print("Total Withdrawn IPO Value:", summary.max(), "million")

Company Type: Other
Total Withdrawn IPO Value: 940.445 million


In [109]:
highest_company_type = summary.idxmax()
highest_value = summary.max()

print(
    f"The company type with the highest total withdrawn IPO value "
    f"is {highest_company_type}, with a total value of "
    f"${highest_value:,.2f} million."
)

The company type with the highest total withdrawn IPO value is Other, with a total value of $940.45 million.


## Q2

In [113]:
def find_col(df, *keywords):
    """Return the first column whose name contains all keywords (case-insensitive)."""
    for c in df.columns:
        if all(k in str(c).lower() for k in keywords):
            return c
    raise KeyError(f"No column with {keywords}. Columns: {list(df.columns)}")


def load_ipo_list(url="https://www.iposcoop.com/2025-pricings/"):
    """Step 1: download the 2025 IPO pricings table (expect ~231 rows)."""
    html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30).text
    tables = pd.read_html(StringIO(html))
    ipo = max(tables, key=len).copy()              # the pricings table is the biggest one
    ipo.columns = [str(c).strip() for c in ipo.columns]
    return ipo


def filter_ipos(ipo):
    """Step 2: offer date before 1 Sep 2025 and non-zero return (expect ~148 rows)."""
    date_col = find_col(ipo, "offer", "date")
    sym_col = find_col(ipo, "symbol")
    ret_col = find_col(ipo, "return")

    ipo = ipo.copy()
    ipo[date_col] = pd.to_datetime(ipo[date_col], errors="coerce")
    ipo["ret_pct"] = pd.to_numeric(
        ipo[ret_col].astype(str).str.replace("%", "", regex=False).str.replace(",", "", regex=False),
        errors="coerce",
    )
    out = ipo[(ipo[date_col] < CUTOFF) & (ipo["ret_pct"] != 0)].copy()   # NaN != 0 -> kept
    out["Ticker"] = out[sym_col].astype(str).str.strip().str.upper()
    out = out[out["Ticker"].notna() & (out["Ticker"] != "NAN")]
    out = out.rename(columns={date_col: "OfferDate"})
    print(f"IPOs after filtering: {len(out)}   (expected ~148)")
    return out.reset_index(drop=True)

In [116]:
def download_prices(tickers, start="2025-01-01", end="2026-09-12", pause=0.3):
    """Step 3: daily OHLCV per ticker. `end` is exclusive, so 09-12 includes 09-11.
    Tickers missing on Yahoo (delisted / not found) are skipped and reported."""
    frames, missing = [], []
    for t in tickers:
        time.sleep(pause)          # gentle on Yahoo's rate limits
        try:
            h = yf.Ticker(t).history(start=start, end=end, auto_adjust=True)
        except Exception:
            h = pd.DataFrame()
        if h is None or h.empty:
            missing.append(t)
            continue
        h = h.reset_index()
        h["Date"] = pd.to_datetime(h["Date"]).dt.tz_localize(None).dt.normalize()
        h["Ticker"] = t
        frames.append(h[["Date", "Ticker", "Open", "High", "Low", "Close", "Volume"]])
    stocks_df = pd.concat(frames, ignore_index=True)
    print(f"Tickers with data: {stocks_df['Ticker'].nunique()} (expected ~134); missing: {len(missing)}")
    return stocks_df, missing

In [127]:
!pip install yfinance


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [129]:
import os
import time
from io import StringIO

import numpy as np
import pandas as pd
import requests
import yfinance as yf
from IPython.display import display

AS_OF = "2026-09-11"           # analysis date for Q2
CUTOFF = "2025-09-01"          # IPOs before this date
RISK_FREE = 0.05
DAYS_PER_MONTH = 21

In [131]:
def add_q2_features(stocks_df, rf=RISK_FREE):
    """Step 4-5: growth_252d, volatility, Sharpe. Computed PER TICKER so windows never
    cross from one stock into the next."""
    df = stocks_df.sort_values(["Ticker", "Date"]).reset_index(drop=True)
    g = df.groupby("Ticker")["Close"]
    df["growth_252d"] = df["Close"] / g.shift(252)
    # formula as specified in the task (std of the Close price, not of returns)
    df["volatility"] = g.transform(lambda s: s.rolling(30).std()) * np.sqrt(252)
    df["Sharpe"] = (df["growth_252d"] - rf) / df["volatility"]
    return df


def q2_answer(stocks_df):
    df = add_q2_features(stocks_df)
    snap = df[df["Date"] == AS_OF]
    stats = snap[["growth_252d", "volatility", "Sharpe"]].describe()
    print("\n=== Q2: describe() on", AS_OF, "===")
    print(stats)
    print(f"\nStocks that reached the 252-day milestone: {snap['growth_252d'].notna().sum()}")
    print(f"growth_252d  median={snap['growth_252d'].median():.4f}  mean={snap['growth_252d'].mean():.4f}")
    print(f"MEDIAN SHARPE RATIO: {snap['Sharpe'].median():.4f}")
    return df, snap

In [133]:
ipo_raw = load_ipo_list()
print(f"Raw IPO rows: {len(ipo_raw)} (expected ~231)")
display(ipo_raw.head())
ipo = filter_ipos(ipo_raw)

Raw IPO rows: 231 (expected ~231)


,Company,Symbol,Industry,Offer Date,Shares (millions),Offer Price,1st Day Close,Current Price,Return,SCOOP Rating
0,Vine Hill Capital Investment Corp. II,VHCPU,Blank Check,12/18/2025,20.0,$10.00,$0.00,$0.00,0.00%,S/O
1,Andersen Group,ANDG,Consumer Services,12/17/2025,11.0,$16.00,$23.50,$23.79,48.69%,S/O
2,Medline Inc.,MDLN,Health Care,12/17/2025,216.0,$29.00,$41.00,$43.83,51.14%,S/O
3,Wealthfront Corp.,WLTH,Financials,12/12/2025,34.6,$14.00,$14.19,$8.49,-39.36%,S/O
4,Lumexa Imaging Holdings,LMRI,Health Care,12/11/2025,25.0,$18.50,$18.52,$14.80,-20.00%,S/O


IPOs after filtering: 146   (expected ~148)


In [135]:
CACHE = "stocks_df.parquet"

if os.path.exists(CACHE):
    stocks_df = pd.read_parquet(CACHE)
    missing = sorted(set(ipo["Ticker"]) - set(stocks_df["Ticker"]))
    print(f"Loaded cache: {stocks_df['Ticker'].nunique()} tickers")
else:
    stocks_df, missing = download_prices(ipo["Ticker"].unique().tolist())
    stocks_df.to_parquet(CACHE)

print("Missing on Yahoo:", missing)

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: MJID"}}}
$MJID: possibly delisted; no timezone found
$DLXY: possibly delisted; no price data found  (1d 2025-01-01 -> 2026-09-12)
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: EMPG"}}}
$EMPG: possibly delisted; no timezone found
$CAEP: possibly delisted; no timezone found
$PTNM: possibly delisted; no timezone found
$AHL: possibly delisted; no timezone found
$CEPT: possibly delisted; no timezone found
$SDM: possibly delisted; no timezone found
$TBH: possibly delisted; no timezone found
$WGRX: possibly delisted; no timezone found
$AGH: possibly delisted; no timezone found
$EPWK: possibly delisted; no timezone found
$MTSR: possibly delisted; no timezone found
$SKBL: possibly delisted; no timezone found
$MCTR: possibly delisted; no timezone found


Tickers with data: 131 (expected ~134); missing: 15
Missing on Yahoo: ['MJID', 'DLXY', 'EMPG', 'CAEP', 'PTNM', 'AHL', 'CEPT', 'SDM', 'TBH', 'WGRX', 'AGH', 'EPWK', 'MTSR', 'SKBL', 'MCTR']


In [137]:
q2_df, q2_snap = q2_answer(stocks_df)
display(q2_snap[["growth_252d", "volatility", "Sharpe"]].describe())


=== Q2: describe() on 2026-09-11 ===
       growth_252d  volatility      Sharpe
count   129.000000  130.000000  129.000000
mean      1.065215   27.725051         inf
std       3.059657   67.095064         NaN
min       0.001005    0.000000   -0.040147
25%       0.149038    2.325895    0.011942
50%       0.596026    7.927899    0.048009
75%       1.040233   23.659042    0.134248
max      33.638270  627.698106         inf

Stocks that reached the 252-day milestone: 129
growth_252d  median=0.5960  mean=1.0652
MEDIAN SHARPE RATIO: 0.0480


,growth_252d,volatility,Sharpe
count,129.000000,130.000000,129.000000
mean,1.065215,27.725051,inf
std,3.059657,67.095064,NaN
min,0.001005,0.000000,-0.040147
25%,0.149038,2.325895,0.011942
50%,0.596026,7.927899,0.048009
75%,1.040233,23.659042,0.134248
max,33.638270,627.698106,inf


## Q3. Best fixed holding period (1-12 months)

In [140]:
def add_future_growth(df, months=range(1, 13), days_per_month=DAYS_PER_MONTH):
    df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)
    g = df.groupby("Ticker")["Close"]
    for m in months:
        df[f"future_growth_{m}_m"] = g.shift(-days_per_month * m) / df["Close"]
    return df


def q3_answer(stocks_df):
    df = add_future_growth(stocks_df)
    cols = [f"future_growth_{m}_m" for m in range(1, 13)]

    # entry point = first available trading day per ticker
    min_date = df.groupby("Ticker", as_index=False)["Date"].min()
    # inner join isolates the returns measured from each stock's first-day close
    entry = min_date.merge(df[["Ticker", "Date"] + cols], on=["Ticker", "Date"], how="inner")

    stats = entry[cols].describe()
    print("\n=== Q3: describe() of future growth from IPO day ===")
    print(stats.T[["count", "mean", "50%", "75%", "max"]])

    best_col = stats.loc["50%"].idxmax()
    best_month = int(best_col.split("_")[2])
    best_val = stats.loc["50%", best_col]
    print(f"\nOPTIMAL HOLDING PERIOD: {best_month} month(s), median growth = {best_val:.4f}")
    print("Mean vs median (outlier check):")
    print(pd.DataFrame({"mean": stats.loc["mean"], "median": stats.loc["50%"]}))
    return entry, stats, best_month, best_val

In [142]:
q3_entry, q3_stats, best_m, best_v = q3_answer(stocks_df)
display(q3_stats)


=== Q3: describe() of future growth from IPO day ===
                    count       mean       50%       75%           max
future_growth_1_m   131.0  96.475452  0.930622  1.114924  12500.000279
future_growth_2_m   130.0  68.494523  0.891684  1.171005   8750.000196
future_growth_3_m   130.0  51.642359  0.827951  1.139366   6562.500147
future_growth_4_m   130.0  22.872042  0.736039  1.197698   2837.500063
future_growth_5_m   130.0  54.660308  0.695853  1.091084   6975.000156
future_growth_6_m   130.0  64.504145  0.733385  1.046022   8250.000184
future_growth_7_m   130.0  59.722684  0.667972  1.018706   7625.000170
future_growth_8_m   130.0  21.624139  0.603984  1.037199   2687.500060
future_growth_9_m   130.0  20.163352  0.584421  1.034648   2475.000055
future_growth_10_m  130.0  11.346041  0.533487  1.036785   1325.000030
future_growth_11_m  130.0   8.971898  0.487132  1.122730   1031.250023
future_growth_12_m  129.0   5.868163  0.508333  1.042121    633.250012

OPTIMAL HOLDING PERIOD

,future_growth_1_m,future_growth_2_m,future_growth_3_m,future_growth_4_m,future_growth_5_m,future_growth_6_m,future_growth_7_m,future_growth_8_m,future_growth_9_m,future_growth_10_m,future_growth_11_m,future_growth_12_m
count,131.000000,130.000000,130.000000,130.000000,130.000000,130.000000,130.000000,130.000000,130.000000,130.000000,130.000000,129.000000
mean,96.475452,68.494523,51.642359,22.872042,54.660308,64.504145,59.722684,21.624139,20.163352,11.346041,8.971898,5.868163
std,1092.037636,767.322855,575.469311,248.776445,611.660280,723.482269,668.665749,235.633687,216.992585,116.157854,90.379471,55.711000
min,0.072941,0.106197,0.090147,0.063927,0.032245,0.022075,0.013229,0.015130,0.014675,0.011947,0.008681,0.004833
25%,0.709910,0.582405,0.449793,0.404543,0.431361,0.315944,0.254490,0.200089,0.180241,0.182353,0.169508,0.164831
50%,0.930622,0.891684,0.827951,0.736039,0.695853,0.733385,0.667972,0.603984,0.584421,0.533487,0.487132,0.508333
75%,1.114924,1.171005,1.139366,1.197698,1.091084,1.046022,1.018706,1.037199,1.034648,1.036785,1.122730,1.042121
max,12500.000279,8750.000196,6562.500147,2837.500063,6975.000156,8250.000184,7625.000170,2687.500060,2475.000055,1325.000030,1031.250023,633.250012


## Q4. RSI < 30 strategy¶

In [145]:
def q4_answer(threshold=30, start="2000-01-01", end="2025-06-01"):
    import gdown
    file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
    gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)
    df = pd.read_parquet("data.parquet", engine="pyarrow")

    # be tolerant about how Date / RSI are named
    if "Date" not in df.columns and "date" not in df.columns:
        df = df.reset_index()
    date_col = next(c for c in df.columns if str(c).lower() == "date")
    rsi_col = next(c for c in df.columns if str(c).lower() == "rsi")
    df[date_col] = pd.to_datetime(df[date_col])

    selected_df = df[
        (df[date_col] >= start) & (df[date_col] <= end) & (df[rsi_col] < threshold)
    ].dropna(subset=["growth_future_30d"])

    net_income = 1000 * (selected_df["growth_future_30d"] - 1).sum()
    print("\n=== Q4: RSI strategy ===")
    print(f"Trades:            {len(selected_df):,}   (expected ~5,206)")
    print(f"Avg 30d return:    {(selected_df['growth_future_30d'] - 1).mean():.2%}   (expected ~1.26%)")
    print(f"Win rate:          {(selected_df['growth_future_30d'] > 1).mean():.2%}   (expected ~55.13%)")
    print(f"NET INCOME:        ${net_income:,.0f}  ->  {net_income / 1000:,.1f} thousand dollars")
    return selected_df, net_income / 1000

In [149]:
%pip install gdown

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [151]:
q4_trades, q4_kusd = q4_answer()
print(f"\nAnswer for Q4: {q4_kusd:,.1f} ($ thousands)")

Downloading...
From (original): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-
From (redirected): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-&confirm=t&uuid=6de704b6-8dc8-4317-96ac-097a66aa1169
To: C:\Users\Tooter\8. Stock Analysis\data.parquet
100%|██████████| 130M/130M [00:15<00:00, 8.26MB/s] 



=== Q4: RSI strategy ===
Trades:            5,206   (expected ~5,206)
Avg 30d return:    1.26%   (expected ~1.26%)
Win rate:          55.13%   (expected ~55.13%)
NET INCOME:        $65,806  ->  65.8 thousand dollars

Answer for Q4: 65.8 ($ thousands)


## Q5. Ideas to raise IPO strategy profitability (exploratory)¶

In [154]:
def q5_delayed_entry(stocks_df, offsets=(0, 1, 5, 10, 21, 42, 63), hold_months=6):
    """Idea 1: don't buy on day 1. Buy N trading days after listing and hold a fixed period.
    Compares median/mean/win-rate across entry delays."""
    df = stocks_df.sort_values(["Ticker", "Date"]).reset_index(drop=True)
    h = DAYS_PER_MONTH * hold_months
    df["fg"] = df.groupby("Ticker")["Close"].shift(-h) / df["Close"]
    df["day_n"] = df.groupby("Ticker").cumcount()
    rows = []
    for d in offsets:
        s = df.loc[df["day_n"] == d, "fg"].dropna()
        rows.append({"entry_day": d, "n": len(s), "median": s.median(),
                     "mean": s.mean(), "win_rate": (s > 1).mean()})
    out = pd.DataFrame(rows)
    print(f"\n=== Q5 idea 1: delayed entry, hold {hold_months} months ===")
    print(out.round(4))
    return out


def q5_classifier(stocks_df, obs_days=5, hold_months=6):
    """Idea 2: predict which IPOs will have positive forward return using only information
    available after `obs_days` of trading (early momentum, volatility, range, volume).
    Time-ordered train/test split to avoid look-ahead. NOTE: ~130 samples -> treat as a
    prototype; add features (offer price, first-day pop, sector, underwriter, market
    regime, lock-up date, float) before trusting it."""
    from sklearn.ensemble import GradientBoostingClassifier
    from sklearn.metrics import precision_score, accuracy_score

    df = stocks_df.sort_values(["Ticker", "Date"]).reset_index(drop=True)
    df["day_n"] = df.groupby("Ticker").cumcount()
    h = DAYS_PER_MONTH * hold_months

    recs = []
    for t, g in df.groupby("Ticker"):
        g = g.reset_index(drop=True)
        if len(g) < obs_days + h + 1:
            continue
        early = g.iloc[: obs_days + 1]
        entry_px = g.loc[obs_days, "Close"]
        exit_px = g.loc[obs_days + h, "Close"]
        recs.append({
            "Ticker": t,
            "start": g.loc[0, "Date"],
            "ret_early": entry_px / g.loc[0, "Open"] - 1,          # early momentum
            "vol_early": early["Close"].pct_change().std(),
            "range_early": ((early["High"] - early["Low"]) / early["Close"]).mean(),
            "volume_trend": early["Volume"].iloc[-1] / max(early["Volume"].iloc[0], 1),
            "fwd_growth": exit_px / entry_px,
        })
    data = pd.DataFrame(recs).sort_values("start").reset_index(drop=True)
    data["y"] = (data["fwd_growth"] > 1).astype(int)
    feats = ["ret_early", "vol_early", "range_early", "volume_trend"]
    data = data.dropna(subset=feats)

    split = int(len(data) * 0.7)
    train, test = data.iloc[:split], data.iloc[split:]
    model = GradientBoostingClassifier(n_estimators=100, max_depth=2, learning_rate=0.05, random_state=0)
    model.fit(train[feats], train["y"])
    pred = model.predict(test[feats])

    print("\n=== Q5 idea 2: classifier (time-ordered split) ===")
    print(f"train={len(train)}  test={len(test)}  base win rate (test)={test['y'].mean():.2%}")
    print(f"accuracy={accuracy_score(test['y'], pred):.2%}  "
          f"precision={precision_score(test['y'], pred, zero_division=0):.2%}")
    picked = test[pred == 1]
    print(f"median growth, all test IPOs: {test['fwd_growth'].median():.4f}")
    if len(picked):
        print(f"median growth, model picks:   {picked['fwd_growth'].median():.4f}  (n={len(picked)})")
    print(pd.Series(model.feature_importances_, index=feats).sort_values(ascending=False).round(3))
    return model, data

In [156]:
delayed = q5_delayed_entry(stocks_df)
display(delayed.round(4))


=== Q5 idea 1: delayed entry, hold 6 months ===
   entry_day    n  median     mean  win_rate
0          0  130  0.7334  64.5041    0.2846
1          1  130  0.7343  58.6904    0.2923
2          5  130  0.7324  64.0051    0.3538
3         10  130  0.7447  58.4296    0.3462
4         21  130  0.7989   0.9284    0.3077
5         42  130  0.7192   0.8791    0.3000
6         63  130  0.7681   1.1366    0.3615


,entry_day,n,median,mean,win_rate
0,0,130,0.7334,64.5041,0.2846
1,1,130,0.7343,58.6904,0.2923
2,5,130,0.7324,64.0051,0.3538
3,10,130,0.7447,58.4296,0.3462
4,21,130,0.7989,0.9284,0.3077
5,42,130,0.7192,0.8791,0.3000
6,63,130,0.7681,1.1366,0.3615


In [158]:
model, q5_data = q5_classifier(stocks_df)


=== Q5 idea 2: classifier (time-ordered split) ===
train=91  test=39  base win rate (test)=33.33%
accuracy=43.59%  precision=15.38%
median growth, all test IPOs: 0.7327
median growth, model picks:   0.4510  (n=13)
range_early     0.308
volume_trend    0.261
ret_early       0.228
vol_early       0.202
dtype: float64
